# Logistic Regression Project (Predict Ad click)

In this notebook we will use `Logistic Regression` to indicating whether or not a particular internet user clicked on an Advertisement. We will try to create a model that will predict whether or not they will click on an ad based off the features of that user.

This data set contains the following features:

* '`Daily Time Spent on Site`': consumer time on site in minutes
* '`Age`': customer age in years
* '`Area Income`': Avg. Income of geographical area of consumer
* '`Daily Internet Usage`': Avg. minutes a day consumer is on the internet
* '`Ad Topic Line`': Headline of the advertisement
* '`City`': City of consumer
* '`Male`': Whether or not consumer was male
* '`Country`': Country of consumer
* '`Timestamp`': Time at which consumer clicked on Ad or closed window
* '`Clicked on Ad`': 0 or 1 indicated clicking on Ad

## Get the Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
df = pd.read_csv("data/advertising.csv")
df

# 1. Exploratory Data Analysis

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df['Clicked on Ad'].value_counts()

In [ ]:
sns.pairplot(df, hue='Clicked on Ad')

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", vmin=-1)

# 2. Prepare Data for Logistic Regression



#### 1. Feature engineering


In [ ]:
df.head()

In [ ]:
df['Ad Topic Line'].nunique()

In [ ]:
df['Ad Topic Line'].str.split(" ")

In [ ]:
lista_palabras = []
for topic in df['Ad Topic Line']:
    for palabra in topic.split(" "):
        lista_palabras.append(palabra.lower())

In [ ]:
pd.Series(lista_palabras).value_counts()

In [ ]:
df['Topic_solution'] = np.where(df['Ad Topic Line'].str.contains("solution"), 1, 0)
df.head()

In [ ]:
df['Topic_solution'].value_counts()

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

In [ ]:
df['City_encoded'] = encoder.fit_transform(df['City'])
df

In [ ]:
df['City_encoded'].nunique()/len(df)

### OneHotEncoding

In [ ]:
pd.get_dummies(df['Country'])

In [ ]:
encoder = LabelEncoder()
df['Country_encoded'] = encoder.fit_transform(df['Country'])
df

In [ ]:
df['Country'].nunique()

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

In [ ]:
df['month'] = df['Timestamp'].dt.month
df['month_dau'] = df['Timestamp'].dt.day
df['week_day'] = df['Timestamp'].dt.weekday
df['hour'] = df['Timestamp'].dt.hour
df.head()

In [ ]:
df.groupby('Topic_solution')['Clicked on Ad'].mean()

In [ ]:
df.groupby('Topic_solution')['Clicked on Ad'].value_counts()

In [ ]:
df.groupby('month')['Clicked on Ad'].mean()

In [ ]:
df.groupby('hour')['Clicked on Ad'].mean()

In [ ]:
df.groupby('hour')['Clicked on Ad'].value_counts(normalize=True)

In [ ]:
df.groupby('month_dau')['Clicked on Ad'].mean()

In [ ]:
df.groupby('week_day')['Clicked on Ad'].mean()

In [ ]:
plt.figure(figsize=(15,10))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", vmin=-1)


#### 2. Train test split


In [ ]:
df.columns

In [ ]:
X = df[['Daily Time Spent on Site', 'Age', 'Area Income',
       'Daily Internet Usage', 'Male', 'Topic_solution', 'month',
       'month_dau', 'week_day', 'hour']]
# X = df[['Daily Time Spent on Site','Daily Internet Usage']]
y = df['Clicked on Ad']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
df['Country_encoded'].hist()


#### 3. StandardScaler()


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scal = scaler.fit_transform(X_train)
X_test_scal = scaler.transform(X_test)

In [ ]:
scaler.inverse_transform(X_test_scal)

# 3. Implement a Logistic Regression in Scikit-Learn and predict. Use cross validation.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn import model_selection

In [ ]:
model = LogisticRegression(max_iter=6000)
model.fit(X, y)

In [ ]:
predictions = model.predict(X)
print(predictions)

In [ ]:
predicions_proba = model.predict_proba(X)
print(np.round(np.array(predicions_proba), 2))
#predicions_proba

In [ ]:
model.score(X, y)

In [ ]:
model.classes_

In [ ]:
validation_size = 0.20
seed = 7
X_train, X_test, Y_train, Y_test = model_selection.train_test_split(X,
                                                                    y,
                                                                    test_size=validation_size,
                                                                    random_state=seed)

In [ ]:
name='Logistic Regression'
kfold = model_selection.KFold(n_splits=10) #Parte los datos en 10 trozos para usar validación cruzada / cross validation
cv_results = model_selection.cross_val_score(model, X_train, Y_train, cv=kfold, scoring='accuracy')

msg = "%s: %f (%f)" % (name, cv_results.mean(), cv_results.std())
print(cv_results)
print(msg)

# 4. Evaluation


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, classification_report


#### 1. Confusion Matrix


In [ ]:
predictions = model.predict(X_test)
print(accuracy_score(Y_test, predictions))
predictions

In [ ]:
acierto = accuracy_score(Y_test, predictions)

error = 1 - acierto
print("Acierto:", round(acierto*100, 2), "%")
print("Error:", round(error*100, 2), "%")

In [ ]:
c_matrix = confusion_matrix(Y_test, predictions)

In [ ]:
sns.heatmap(c_matrix, annot=True);

In [ ]:
sns.heatmap(confusion_matrix(Y_test, predictions, normalize='true'), annot=True, 
            fmt='.2%', cmap='Blues');


#### 2. Precision


In [ ]:
print(classification_report(y_test, predictions))



#### 3. Recall


In [ ]:
print(classification_report(y_test, predictions))



#### 4. F1 Score


In [ ]:
print(classification_report(y_test, predictions))



#### 5. ROC curve

In [ ]:
from sklearn.metrics import roc_curve

In [ ]:
fpr, tpr, _ = roc_curve(Y_test, predicions_proba)



#### 6. P-R curve

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, _ = precision_recall_curve(Y_test, predicions_proba)


ValueError: Found input variables with inconsistent numbers of samples: [200, 1000]